# 도움 요청 말 감지 모델 (3차: Pi 마이크 실제 녹음(R)을 양성 학습에 추가)
순서: ① 드라이브 연결 → ② 설정·데이터 → ③ 특징·모델 → ④ 학습·평가 함수 → ⑤ 실제 녹음 개발용 세트 → ⑤-2 Pi 녹음(R 학습, C 보정) → ⑥ 실행(2차 모델 재평가 → 3차 학습) → ⑥-2 Pi 보정 녹음 점검 → ⑦ ONNX
- 3차의 핵심: Pi 마이크로 녹음한 R(학습용, 살려·사람 살려·119 위주)을 A~D 양성의 일부(P_REAL_POS)로 섞는다. 모델 고르기는 2차와 같은 6_own 기준이라 2차와 바로 비교된다.
- 시험용 P·N 녹음은 이 노트북에서 읽지 않는다(최종 평가 전용). C는 보정 점검(⑥-2)에만 쓴다.
- GPU(T4)로 '모두 실행'. 끝나면 '런타임 연결 해제 및 삭제'.


In [ ]:
# ① 드라이브 연결 (Colab에서만)
import os, sys
if os.path.isdir("/content"):
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# ② 설정과 데이터 읽기
import csv, json, time, math, glob, shutil, hashlib, collections, subprocess, unicodedata
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

MODE = os.environ.get("KWS_MODE", "full")             # "full": 실제 학습·평가 (끝난 실험은 건너뜀) / "smoke": 몇 분짜리 동작 확인(점수는 의미 없음)
ROOT = os.environ.get("KWS_ROOT", "/content/drive/MyDrive/LAB")
LOCAL = os.environ.get("KWS_LOCAL", "/content/kws_local")
OUT = os.path.join(ROOT, "kws")                       # 결과 저장(드라이브)
WORKERS = int(os.environ.get("KWS_WORKERS", "2"))
SMOKE = MODE != "full"

SR = 16000
WIN = int(3.5 * SR)                                   # 모델 입력 3.5초 (도움 요청 말은 90%가 2.7초 이하)
N_FFT, HOP, N_MELS, FMIN, FMAX = 512, 160, 40, 150.0, 7600.0
N_FRAMES = 1 + (WIN - N_FFT) // HOP
CLASSES = ["none", "A", "B", "C", "D", "E"]           # E = 신고·와주세요 계열(알림 대상 아님, 설정으로 바꿀 수 있음)
ALERT = [1, 2, 3, 4]
ROLE2CLS = {"pos_A": 1, "pos_B": 2, "pos_C": 3, "pos_D": 4, "req_E": 5, "neg_dog": 0, "neg_med": 0, "neg_fall": 0}
BATCH = 32 if SMOKE else 256
S1_STEPS = int(os.environ.get("KWS_S1", 20 if SMOKE else 3000))     # 후보 비교용 학습 길이
S2_STEPS = int(os.environ.get("KWS_S2", 30 if SMOKE else 9000))     # 최종 후보 학습 길이
EVAL_EVERY = 20 if SMOKE else 500
CKPT_EVERY = 10 if SMOKE else 250
AMB_MAX, SP_MAX = (300, 300) if SMOKE else (8000, 8000)   # 메모리에 올릴 배경·일상 말소리 창 수
NEG_MIN, POS_MIN = (2, 2) if SMOKE else (240, 60)     # 연속 평가용 소리 길이(분): 오경보를 더 믿을 만하게 재려고 오답 쪽을 4시간으로
HOP_S = 0.25                                          # 연속 평가에서 판단 간격(초)
FA_TARGET = 1.0                                       # 시간당 오경보 목표(합성 연속 소리 기준)
VER = "r3"                                            # 3차 학습 표시(결과 이름 r3_*)
R2_STEPS = int(os.environ.get("KWS_R2", 20 if SMOKE else 3000))     # 1차에서 3000단계가 9000단계보다 실제 녹음에 강했음
CANDS2 = os.environ.get("KWS_CANDS2", "tcresnet").split(",")          # 2차 최선이 tcresnet이라 3차는 하나만(코랩 단위 절약)
P_REAL_POS = float(os.environ.get("KWS_P_REAL_POS", "0.2"))          # A~D 양성을 뽑을 때 Pi 실제 녹음(R)에서 뽑는 비율
DEV_EVERY = int(os.environ.get("KWS_DEV_EVERY", 20 if SMOKE else 500))   # 이 단계마다 실제 녹음으로 평가해서 가장 좋은 시점을 저장
DEV_FA_TARGET = 1.0                                   # 실제 녹음에서 허용할 시간당 오경보(이 안에서 재현율이 가장 높은 기준을 고름)
GATE_REL, GATE_ABS = 15.0, -65.0                      # 소리 크기 게이트: 최근 1.5초에 바닥보다 15dB 이상(그리고 -65dBFS 이상) 큰 소리가 있을 때만 판단
P_REAL_BG, P_REAL_SP = 0.25, 0.35                     # 학습 때 실제 녹음(학습용 폴더)을 배경·일상 말소리로 쓰는 비율
REAL_TR = None                                        # ⑤번 셀에서 채움
DEV = None                                            # ⑤번 셀에서 채움(실제 녹음 개발용 세트)
SEED = 0
device = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUT, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)
print("MODE:", MODE, "| device:", device, "| torch", torch.__version__)
print("결과 저장 폴더: %s (실제 학습은 runs_full, 동작 확인은 runs_smoke)" % os.path.join(OUT, "runs_smoke" if SMOKE else "runs_full"))
if SMOKE:
    print("※ 지금은 동작 확인 모드(smoke)입니다. 학습이 20단계뿐이라 점수는 의미가 없습니다. 실제 학습은 MODE를 \"full\"로 바꾸세요.")
with open(os.path.join(OUT, "gpu_log.csv"), "a", encoding="utf-8") as f:      # GPU 사용 가능 여부 기록(한도 패턴 확인용)
    f.write("%s,%s,%s\n" % (time.strftime("%Y-%m-%d %H:%M:%S"), MODE, torch.cuda.get_device_name(0) if device == "cuda" else "cpu"))

def find_dir(name, required=True):
    for base in (ROOT, os.path.dirname(ROOT)):
        p = os.path.join(base, name)
        if os.path.isdir(p):
            return p
    if required:
        raise FileNotFoundError("드라이브에서 폴더를 찾지 못했습니다: " + name)
    return None

def localize(src_dir, names, sub):
    """드라이브 파일을 Colab 디스크로 복사(이미 있으면 건너뜀)"""
    dst = os.path.join(LOCAL, sub); os.makedirs(dst, exist_ok=True)
    for n in names:
        s, d = os.path.join(src_dir, n), os.path.join(dst, n)
        if not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(s):
            shutil.copy(s, d)
    return dst

def read_csv(p):
    with open(p, encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

# ---- 도움 요청 클립(help_data, 있으면 help_data2도) ----
def load_help(src, tag):
    names = [os.path.basename(p) for p in glob.glob(os.path.join(src, "meta_*.csv")) + glob.glob(os.path.join(src, "clips_*.npy"))]
    d = localize(src, names, tag)
    rows = []
    for sp in ("Training", "Validation"):
        mp = os.path.join(d, "meta_%s.csv" % sp)
        if not os.path.exists(mp):
            continue
        shards = {}
        for r in read_csv(mp):
            if r["role"] not in ROLE2CLS:
                continue
            f = r["shard_file"]
            if f not in shards:
                shards[f] = np.load(os.path.join(d, f), mmap_mode="r")      # 메모리에 다 올리지 않고 디스크에서 읽음
            r["arr"], r["o"], r["n"], r["split"], r["cls_id"] = shards[f], int(r["offset"]), int(r["length"]), sp, ROLE2CLS[r["role"]]
            rows.append(r)
    return rows

def clip16(r):
    return r["arr"][r["o"]:r["o"] + r["n"]]

def speech_span(x16):
    """클립 안에서 말소리가 있는 구간(시작·끝 샘플). 20ms 단위 크기로 판단"""
    fr = 320
    n = len(x16) // fr
    if n < 5:
        return 0, len(x16)
    e = 10 * np.log10((x16[:n * fr].astype(np.float32).reshape(n, fr) ** 2).mean(1) + 1.0)
    thr = max(e.max() - 30.0, np.percentile(e, 10) + 3.0)
    idx = np.where(e > thr)[0]
    if len(idx) == 0:
        return 0, len(x16)
    return max(int(idx[0]) * fr - 800, 0), min((int(idx[-1]) + 1) * fr + 800, len(x16))

def stem_id(stem):
    try:
        return int(stem.rsplit("_", 1)[1])
    except Exception:
        return int(hashlib.md5(stem.encode()).hexdigest()[:8], 16)

def is_holdout(r):
    """파일 번호 500개 단위 묶음의 10%를 검증용으로 뗀다(가까운 번호는 같은 사람일 수 있어서)"""
    return (((stem_id(r["stem"]) // 500) * 2654435761) % (2 ** 32)) % 10 == 0

rows = load_help(find_dir("help_data"), "help_data")
extra = find_dir("help_data2", required=False)
if extra:
    seen = {r["stem"] for r in rows}
    rows += [r for r in load_help(extra, "help_data2") if r["stem"] not in seen]
print("help_data2(추가 데이터):", "있음" if extra else "없음 → E 클래스와 추가 양성 없이 진행")

t0 = time.time()
h_tr = set()
for r in rows:
    x = clip16(r)
    r["s0"], r["s1"] = speech_span(x)
    r["md5"] = hashlib.md5(x.tobytes()).hexdigest()
    r["val"] = (r["split"] == "Validation") or is_holdout(r)
    if not r["val"]:
        h_tr.add(r["md5"])
n0 = len(rows)
rows = [r for r in rows if not (r["val"] and r["md5"] in h_tr)]          # 학습과 같은 소리가 검증에 있으면 검증에서 뺌
TR = [r for r in rows if not r["val"]]
VA = [r for r in rows if r["val"]]

def fits(r):
    """말소리가 창(3.5초)에 온전히 들어가는 클립만 양성으로 쓴다. 잘린 말('…와 주세요')은 E 계열과 헷갈리기 때문"""
    return (r["s1"] - r["s0"]) / SR <= WIN / SR - 0.3
print("클립 %d개 (학습 %d / 검증 %d, 겹쳐서 뺀 검증 %d) | 구간 계산 %.0f초" % (len(rows), len(TR), len(VA), n0 - len(rows), time.time() - t0))
for name, rs in (("학습", TR), ("검증", VA)):
    c = collections.Counter(r["role"] for r in rs)
    print("  %s:" % name, ", ".join("%s %d" % (k, c[k]) for k in sorted(c)))
_pos = [r for r in TR if r["cls_id"] in ALERT]
print("  창에 못 담는 긴 양성(%.1f초 초과): 학습 %d개 / %d개 → 양성에서 뺌 | 검증 %d개" % (
    WIN / SR - 0.3, sum(not fits(r) for r in _pos), len(_pos), sum(not fits(r) for r in VA if r["cls_id"] in ALERT)))
print("  말소리 길이(초, 학습 양성): " + " | ".join("%s 중앙 %.1f / 90%% %.1f" % (
    CLASSES[k], np.median(v), np.percentile(v, 90)) for k in ALERT
    for v in [[(r["s1"] - r["s0"]) / SR for r in TR if r["cls_id"] == k]] if v))

# ---- 배경 소리(실내·실외)와 일상 말소리(Zeroth) : 이전에 변환해 둔 4초 창 ----
def load_windows(src, meta_name, cap, seed):
    rs = read_csv(os.path.join(src, meta_name))
    if SMOKE:                                                   # 동작 확인 때는 첫 파일 하나만 복사
        rs = [r for r in rs if r["shard_file"] == rs[0]["shard_file"]]
    pick = np.random.default_rng(seed).permutation(len(rs))[:cap]
    names = sorted({rs[i]["shard_file"] for i in pick})
    d = localize(src, names, "amb")
    out = np.empty((len(pick), 4 * SR), np.int16)
    by = collections.defaultdict(list)
    for j, i in enumerate(pick):
        by[rs[i]["shard_file"]].append((int(rs[i]["row"]), j))
    for nm, lst in by.items():
        a = np.load(os.path.join(d, nm), mmap_mode="r")
        for row, j in sorted(lst):
            out[j] = a[row]
    return out

def load_speech(src, name, cap, seed):
    d = localize(src, [name], "speech")
    a = np.load(os.path.join(d, name), mmap_mode="r")
    pick = np.sort(np.random.default_rng(seed).permutation(len(a))[:cap])
    return np.ascontiguousarray(a[pick])

P16 = find_dir("processed16k"); S16 = find_dir("speech16k")
AMB_TR = load_windows(P16, "meta_Training_ambient.csv", AMB_MAX, 1)
AMB_VA = load_windows(P16, "meta_Validation_ambient.csv", max(AMB_MAX // 4, 100), 2)
SP_TR = load_speech(S16, "speech_train.npy", SP_MAX, 3)
SP_VA = load_speech(S16, "speech_test.npy", max(SP_MAX // 4, 100), 4)
print("배경 창: 학습 %d / 검증 %d | 일상 말소리 창: 학습 %d / 검증(다른 화자) %d" % (len(AMB_TR), len(AMB_VA), len(SP_TR), len(SP_VA)))

In [ ]:
# ③ 학습용 창 만들기 · 마이크 흉내 · 특징 · 모델
def rms(x):
    return float(np.sqrt(np.mean(np.square(x, dtype=np.float64))) + 1e-9)

RAMP = np.linspace(0, 1, 160, dtype=np.float32)
RAMP2 = np.linspace(0, 1, 320, dtype=np.float32)

def place(x, s0, s1, rng):
    """클립의 말소리 끝이 창 끝에서 d초 앞에 오도록 3초 창에 놓는다. 말소리 크기(RMS)를 1로 맞춘다."""
    dmax = min(1.2, WIN / SR - (s1 - s0) / SR - 0.05)
    d = rng.uniform(0.15, dmax) if dmax > 0.2 else 0.15
    end = WIN - int(d * SR)
    a = s1 - end                                  # 창의 0번 위치에 해당하는 클립 위치
    src0, src1 = max(a, 0), min(len(x), a + WIN)
    dst0 = src0 - a
    seg = x[src0:src1].astype(np.float32)
    if len(seg) > 320:
        if src0 > 0:
            seg[:160] *= RAMP
        if src1 < len(x):
            seg[-160:] *= RAMP[::-1]
    w = np.zeros(WIN, np.float32)
    w[dst0:dst0 + len(seg)] = seg
    sp = w[max(dst0, s0 - a, 0):end]
    w /= rms(sp) if len(sp) > 160 else rms(seg)
    return w

def speed(x, s0, s1, r):
    n = int(len(x) / r)
    y = np.interp(np.arange(n) * r, np.arange(len(x)), x).astype(np.float32)
    return y, int(s0 / r), min(int(s1 / r), n)

class Pool:
    """문장별로 고르게, 노인·먼 거리 클립은 더 자주 뽑는 클립 묶음"""
    def __init__(self, rs, frac=1.0):
        if frac < 1.0:
            rs = [r for r in rs if int(r["md5"][:6], 16) / 16777216.0 < frac]
        by = collections.defaultdict(list)
        for r in rs:
            by[r["note"]].append(r)
        self.notes = [by[k] for k in sorted(by)]
        self.cum = []
        for lst in self.notes:
            w = np.array([(2.0 if r.get("generation") == "senior" else 1.0) * (1.5 if r.get("dist") == "원거리" else 1.0) for r in lst])
            self.cum.append(np.cumsum(w) / w.sum())
        p = np.sqrt([len(l) for l in self.notes]) if self.notes else np.array([1.0])
        self.pn = np.cumsum(p / p.sum())
        self.n = len(rs)
    def sample(self, rng):
        k = min(int(np.searchsorted(self.pn, rng.random())), len(self.notes) - 1)
        j = min(int(np.searchsorted(self.cum[k], rng.random())), len(self.notes[k]) - 1)
        return self.notes[k][j]

def make_pools(rs, frac=1.0):
    P = {k: Pool([r for r in rs if r["cls_id"] == k and (k not in ALERT or fits(r))], frac if k in ALERT else 1.0) for k in (1, 2, 3, 4, 5)}
    for role in ("neg_dog", "neg_med", "neg_fall"):
        P[role] = Pool([r for r in rs if r["role"] == role])
    return P

KIND_FRAC = [("A", .094), ("B", .094), ("C", .094), ("D", .094), ("E", .094), ("hn", .218), ("sp", .187), ("bg", .125)]

def gen_item(kind, P, AMB, SPW, rng, train=True, row=None):
    """창 하나: 앞소리 fg(말소리 RMS=1), 배경 bg(RMS=1), 정답, 앞소리·배경 크기"""
    y, fg, meta = 0, None, None
    if kind in ("A", "B", "C", "D", "E", "hn"):
        if row is not None:
            r = row
        elif kind == "hn":
            role = ("neg_dog", "neg_med", "neg_fall")[int(np.searchsorted([0.15, 0.6, 1.0], rng.random()))]
            r = (P[role] if P[role].n else next(P[q] for q in ("neg_med", "neg_fall", "neg_dog") if P[q].n)).sample(rng)
        else:
            r = P["ABCDE".index(kind) + 1].sample(rng)
        x, s0, s1 = clip16(r), r["s0"], r["s1"]
        if train and rng.random() < 0.4:                       # 말 빠르기·높낮이를 ±10% 바꿈
            x, s0, s1 = speed(x, s0, s1, rng.uniform(0.9, 1.1))
        fg, y, meta = place(x, s0, s1, rng), r["cls_id"], r
    elif kind == "sp":                                          # 일상 말소리 조각(정답: 없음). 학습 때 일부는 실제 녹음에서
        use_real = train and REAL_TR is not None and len(REAL_TR) and rng.random() < P_REAL_SP
        w = REAL_TR[int(rng.integers(len(REAL_TR)))] if use_real else SPW[int(rng.integers(len(SPW)))]
        L = int(rng.uniform(1.0, 2.8) * SR)
        st = int(rng.integers(0, len(w) - L + 1))
        x = w[st:st + L].astype(np.float32)
        x[:320] *= RAMP2
        x[-320:] *= RAMP2[::-1]
        fg = place(x, 0, L, rng)
    bg, has = np.zeros(WIN, np.float32), False
    if rng.random() < 0.8:                                      # 실내·실외 배경 소리. 학습 때 일부는 실제 녹음에서
        src = REAL_TR if (train and REAL_TR is not None and len(REAL_TR) and rng.random() < P_REAL_BG) else AMB
        a = src[int(rng.integers(len(src)))]
        o = int(rng.integers(0, len(a) - WIN + 1))
        b = a[o:o + WIN].astype(np.float32)
        bg, has = b / rms(b), True
    if rng.random() < 0.25:                                     # 뒤에서 다른 사람이 말하는 소리(TV 등)
        s = SPW[int(rng.integers(len(SPW)))]
        o = int(rng.integers(0, len(s) - WIN + 1))
        b = s[o:o + WIN].astype(np.float32)
        b /= rms(b)
        bg = bg + b * 10 ** (-rng.uniform(0, 12) / 20) if has else b
        bg, has = bg / rms(bg), True
    if fg is None:
        fg, lf = np.zeros(WIN, np.float32), 0.0
        lb = 10 ** (rng.uniform(-60, -22) / 20) if has else 0.0
    else:
        Lf = rng.uniform(-45, -15)                              # 말소리 크기(dBFS): 우리 마이크는 1m에서 약 -38
        lf = 10 ** (Lf / 20)
        lb = 10 ** ((Lf - rng.uniform(0, 30)) / 20) if has else 0.0
    return fg, bg, y, lf, lb, meta

class BatchGen(torch.utils.data.IterableDataset):
    def __init__(self, rs, AMB, SPW, bs, seed, frac=1.0):
        self.P, self.AMB, self.SPW, self.seed = make_pools(rs, frac), AMB, SPW, seed
        kinds = []
        for k, f in KIND_FRAC:
            kinds += [k] * int(round(f * bs))
        if not self.P[5].n:
            kinds = ["hn" if k == "E" else k for k in kinds]
        self.kinds = (kinds + ["hn"] * bs)[:bs]
    def __iter__(self):
        wi = torch.utils.data.get_worker_info()
        rng = np.random.default_rng([self.seed, wi.id if wi else 0, int(time.time() * 1000) % (2 ** 31)])
        bs = len(self.kinds)
        while True:
            fg, bg = np.empty((bs, WIN), np.float32), np.empty((bs, WIN), np.float32)
            y, lf, lb = np.empty(bs, np.int64), np.empty(bs, np.float32), np.empty(bs, np.float32)
            for i, k in enumerate(self.kinds):
                fg[i], bg[i], y[i], lf[i], lb[i], _ = gen_item(k, self.P, self.AMB, self.SPW, rng)
            yield torch.from_numpy(fg), torch.from_numpy(bg), torch.from_numpy(y), torch.from_numpy(lf), torch.from_numpy(lb)

def mix_aug(fg, bg, lf, lb, gen=None):
    """앞소리에 방 울림을 넣고 배경과 섞은 뒤, 마이크 잡음·전원 잡음·주파수 특성·찌그러짐을 흉내 낸다"""
    dev = fg.device
    B, T = fg.shape
    U = lambda *s: torch.rand(*s, generator=gen, device=dev)
    N = lambda *s: torch.randn(*s, generator=gen, device=dev)
    sel = (U(B) < 0.5) & (lf > 0)
    if bool(sel.any()):                                         # 방 울림(먼 거리 흉내)
        k, L = int(sel.sum()), int(0.6 * SR)
        t = torch.arange(L, device=dev) / SR
        h = N(k, L) * torch.exp(-6.91 * t / (0.15 + 0.75 * U(k, 1)))
        h[:, :16] = 0
        h = h / h.pow(2).sum(1, keepdim=True).sqrt() * 10 ** (-(-3 + 18 * U(k, 1)) / 20)
        h[:, 0] = 1.0
        n = 1 << (T + L - 1).bit_length()
        src = fg[sel]
        y = torch.fft.irfft(torch.fft.rfft(src, n) * torch.fft.rfft(h, n), n)[:, :T]
        y = y * (src.pow(2).mean(1, keepdim=True).sqrt() / (y.pow(2).mean(1, keepdim=True).sqrt() + 1e-8))
        fg = fg.clone()
        fg[sel] = y
    x = fg * lf[:, None] + bg * lb[:, None]
    x = x + N(B, T) * 10 ** ((-85 + 19 * U(B, 1)) / 20)        # 마이크 자체 잡음
    tt = torch.arange(T, device=dev) / SR
    hum = (U(B, 1) < 0.5).float() * 10 ** ((-85 + 23 * U(B, 1)) / 20)
    ph = 6.2832 * U(B, 1)
    x = x + hum * (torch.sin(6.2832 * 60 * tt + ph) + 0.5 * torch.sin(6.2832 * 120 * tt + 2 * ph))   # 전원 잡음 60/120Hz
    f = torch.fft.rfftfreq(T, 1 / SR).to(dev).clamp(min=50.0)
    tilt = (U(B, 1) < 0.5).float() * (-3 + 6 * U(B, 1))        # 마이크 주파수 특성 기울기(dB/옥타브)
    gain = 10 ** ((tilt * torch.log2(f / 1000.0)).clamp(-12, 12) / 20)
    lp, fc = U(B, 1) < 0.15, 3000 + 4000 * U(B, 1)             # 고음이 깎이는 기기
    gain = torch.where(lp, gain / torch.sqrt(1 + (f / fc) ** 8), gain)
    x = torch.fft.irfft(torch.fft.rfft(x) * gain, T)
    cl = U(B, 1) < 0.12                                         # 찌그러짐(모든 종류에 같은 확률)
    pk = x.abs().amax(1, keepdim=True) + 1e-8
    x = torch.where(cl, x * (1.2 + 1.8 * U(B, 1)) / pk, x)
    return x.clamp(-1, 1)

# ---- 특징: log-mel + 채널별 평균 제거(마이크 특성·크기 차이를 지움) ----
def mel_fb():
    h2m = lambda f: 2595.0 * np.log10(1.0 + f / 700.0)
    m2h = lambda m: 700.0 * (10 ** (m / 2595.0) - 1.0)
    freqs = np.linspace(0, SR / 2.0, N_FFT // 2 + 1)
    pts = m2h(np.linspace(h2m(FMIN), h2m(FMAX), N_MELS + 2))
    fb = np.zeros((N_MELS, len(freqs)), np.float32)
    for i in range(N_MELS):
        l, c, r = pts[i], pts[i + 1], pts[i + 2]
        fb[i] = np.maximum(0.0, np.minimum((freqs - l) / (c - l), (r - freqs) / (r - c)))
    return fb

FB = mel_fb()
HANN = (0.5 - 0.5 * np.cos(2 * np.pi * np.arange(N_FFT) / N_FFT)).astype(np.float32)

class Frontend(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("win", torch.from_numpy(HANN))
        self.register_buffer("fb", torch.from_numpy(FB))
    def forward(self, wav):                                     # (B, T) -> (B, 40, 프레임)
        s = torch.stft(wav.float(), N_FFT, HOP, N_FFT, window=self.win, center=False, return_complex=True)
        L = torch.log(torch.matmul(self.fb, s.real ** 2 + s.imag ** 2) + 1e-10)
        L = L - L.mean(dim=2, keepdim=True)
        return L / (torch.sqrt((L ** 2).mean(dim=(1, 2), keepdim=True)) + 1e-5)

def np_features(x):
    """파이에서 쓸 numpy 특징(위 Frontend와 같은 계산). x: 길이 WIN인 float32"""
    idx = np.arange(N_FFT)[None, :] + HOP * np.arange(1 + (len(x) - N_FFT) // HOP)[:, None]
    s = np.fft.rfft(x[idx].astype(np.float32) * HANN, axis=1)
    L = np.log((s.real ** 2 + s.imag ** 2) @ FB.T + 1e-10).T
    L = L - L.mean(axis=1, keepdims=True)
    return (L / (np.sqrt((L ** 2).mean()) + 1e-5)).astype(np.float32)

def spec_aug(x):
    B, M, Tn = x.shape
    m, t = torch.arange(M, device=x.device)[None, :], torch.arange(Tn, device=x.device)[None, :]
    for _ in range(2):
        w = torch.randint(0, 6, (B, 1), device=x.device)
        s = (torch.rand(B, 1, device=x.device) * (M - w)).long()
        x = x.masked_fill(((m >= s) & (m < s + w))[:, :, None], 0.0)
    w = torch.randint(0, 9, (B, 1), device=x.device)
    s = (torch.rand(B, 1, device=x.device) * (Tn - w)).long()
    return x.masked_fill(((t >= s) & (t < s + w))[:, None, :], 0.0)

# ---- 모델 후보 4개 (입력: (B,1,40,프레임), 출력: 6개 종류 점수) ----
class TCBlock(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1, self.b1 = nn.Conv1d(cin, cout, 9, stride, 4, bias=False), nn.BatchNorm1d(cout)
        self.c2, self.b2 = nn.Conv1d(cout, cout, 9, 1, 4, bias=False), nn.BatchNorm1d(cout)
        self.sc = None if (cin == cout and stride == 1) else nn.Sequential(nn.Conv1d(cin, cout, 1, stride, bias=False), nn.BatchNorm1d(cout))
    def forward(self, x):
        y = self.b2(self.c2(F.relu(self.b1(self.c1(x)))))
        return F.relu(y + (x if self.sc is None else self.sc(x)))

class TCResNet(nn.Module):
    """시간축 합성곱: 주파수 40칸을 채널로 보고 시간 방향으로만 훑는다(CPU에서 빠름)"""
    def __init__(self, ncls=6, width=1.5):
        super().__init__()
        ch = [int(c * width) for c in (16, 24, 32, 48)]
        self.stem = nn.Conv1d(N_MELS, ch[0], 3, 1, 1, bias=False)
        blocks, cin = [], ch[0]
        for c in ch[1:]:
            blocks += [TCBlock(cin, c, 2), TCBlock(c, c, 1)]
            cin = c
        self.body, self.drop, self.fc = nn.Sequential(*blocks), nn.Dropout(0.2), nn.Linear(cin, ncls)
    def forward(self, x):
        return self.fc(self.drop(self.body(self.stem(x.squeeze(1))).mean(2)))

class DSBlock(nn.Sequential):
    def __init__(self, c, stride=1):
        super().__init__(nn.Conv2d(c, c, 3, stride, 1, groups=c, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                         nn.Conv2d(c, c, 1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True))

class DSCNN(nn.Module):
    """깊이별 분리 합성곱: 키워드 인식의 고전적인 작은 구조"""
    def __init__(self, ncls=6, c=96):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, c, 5, 2, 2, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
        self.body = nn.Sequential(DSBlock(c), DSBlock(c), DSBlock(c, 2), DSBlock(c), DSBlock(c))
        self.drop, self.fc = nn.Dropout(0.2), nn.Linear(c, ncls)
    def forward(self, x):
        return self.fc(self.drop(self.body(self.stem(x)).mean((2, 3))))

class SubSpectralNorm(nn.Module):
    def __init__(self, c, S=5):
        super().__init__()
        self.S, self.bn = S, nn.BatchNorm2d(c * S)
    def forward(self, x):
        B, C, Fq, T = x.shape
        return self.bn(x.reshape(B, C * self.S, Fq // self.S, T)).reshape(B, C, Fq, T)

class BCBlock(nn.Module):
    def __init__(self, cin, cout, fstride=1, tdil=1, drop=0.1):
        super().__init__()
        self.trans = cin != cout or fstride != 1
        l = [nn.Conv2d(cin, cout, 1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)] if self.trans else []
        l += [nn.Conv2d(cout, cout, (3, 1), (fstride, 1), (1, 0), groups=cout, bias=False), SubSpectralNorm(cout)]
        self.f2 = nn.Sequential(*l)
        self.f1 = nn.Sequential(nn.Conv2d(cout, cout, (1, 3), 1, (0, tdil), dilation=(1, tdil), groups=cout, bias=False),
                                nn.BatchNorm2d(cout), nn.SiLU(inplace=True), nn.Conv2d(cout, cout, 1, bias=False), nn.Dropout2d(drop))
    def forward(self, x):
        y2 = self.f2(x)
        y = y2 + self.f1(y2.mean(dim=2, keepdim=True))          # 주파수로 평균 낸 시간 흐름을 모든 주파수에 더함
        return F.relu(y if self.trans else y + x)

class BCResNet(nn.Module):
    """BC-ResNet 계열: 주파수 방향 2D + 시간 방향 1D를 섞은 소형 키워드 인식 구조"""
    def __init__(self, ncls=6, tau=3):
        super().__init__()
        c = [int(tau * v) for v in (16, 8, 12, 16, 20, 32)]
        self.head = nn.Sequential(nn.Conv2d(1, c[0], 5, 2, 2, bias=False), nn.BatchNorm2d(c[0]), nn.ReLU(inplace=True))
        self.body = nn.Sequential(
            BCBlock(c[0], c[1]), BCBlock(c[1], c[1]),
            BCBlock(c[1], c[2], 2, 2), BCBlock(c[2], c[2], 1, 2),
            BCBlock(c[2], c[3], 2, 4), BCBlock(c[3], c[3], 1, 4), BCBlock(c[3], c[3], 1, 4), BCBlock(c[3], c[3], 1, 4),
            BCBlock(c[3], c[4], 1, 8), BCBlock(c[4], c[4], 1, 8), BCBlock(c[4], c[4], 1, 8), BCBlock(c[4], c[4], 1, 8))
        self.tail = nn.Sequential(nn.Conv2d(c[4], c[4], 5, 1, (0, 2), groups=c[4], bias=False), nn.BatchNorm2d(c[4]),
                                  nn.Conv2d(c[4], c[5], 1, bias=False), nn.BatchNorm2d(c[5]), nn.ReLU(inplace=True))
        self.fc = nn.Conv2d(c[5], ncls, 1)
    def forward(self, x):
        x = self.tail(self.body(self.head(x)))
        return self.fc(x.mean(dim=(2, 3), keepdim=True)).flatten(1)

class CRNN(nn.Module):
    """합성곱 + 순환층: 말의 순서를 따라가며 본다"""
    def __init__(self, ncls=6, c=32, h=64):
        super().__init__()
        def cb(i, o, s):
            return [nn.Conv2d(i, o, 3, s, 1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True)]
        self.conv = nn.Sequential(*cb(1, c, (2, 2)), *cb(c, c, (2, 2)), *cb(c, c, (2, 1)))
        self.gru = nn.GRU(c * 5, h, batch_first=True, bidirectional=True)
        self.drop, self.fc = nn.Dropout(0.2), nn.Linear(2 * h, ncls)
    def forward(self, x):
        x = self.conv(x)
        y, _ = self.gru(x.permute(0, 3, 1, 2).flatten(2))
        return self.fc(self.drop(y.mean(1)))

ARCH = {"bcresnet": lambda: BCResNet(len(CLASSES), 3), "tcresnet": lambda: TCResNet(len(CLASSES), 1.5),
        "dscnn": lambda: DSCNN(len(CLASSES), 96), "crnn": lambda: CRNN(len(CLASSES))}
FRONT = Frontend().to(device)
for k, fn in ARCH.items():
    print("%-9s 매개변수 %6d개" % (k, sum(p.numel() for p in fn().parameters())))

In [ ]:
# ④ 검증용 창 · 학습 · 연속 소리 평가 함수
def build_val(seed=1234):
    """검증용 창(고정). 모든 실험을 같은 창으로 비교한다"""
    rng, g = np.random.default_rng(seed), torch.Generator().manual_seed(seed)
    P = make_pools(VA)
    hn = [r for r in VA if r["cls_id"] == 0]
    rng.shuffle(hn)
    todo = [("ABCD"[r["cls_id"] - 1], r) for r in VA if r["cls_id"] in ALERT and fits(r)]
    todo += [("E", r) for r in [r for r in VA if r["cls_id"] == 5][:800]]
    todo += [("hn", r) for r in hn[:(60 if SMOKE else 2000)]]
    todo += [("sp", None)] * (60 if SMOKE else 2000) + [("bg", None)] * (30 if SMOKE else 800)
    X = torch.empty((len(todo), WIN), dtype=torch.int16)
    y, kind, gen, dist = np.zeros(len(todo), np.int64), [], [], []
    for i in range(0, len(todo), 128):
        it = [gen_item(k, P, AMB_VA, SP_VA, rng, train=False, row=r) for k, r in todo[i:i + 128]]
        x = mix_aug(torch.from_numpy(np.stack([v[0] for v in it])), torch.from_numpy(np.stack([v[1] for v in it])),
                    torch.tensor([v[3] for v in it], dtype=torch.float32), torch.tensor([v[4] for v in it], dtype=torch.float32), gen=g)
        X[i:i + len(it)] = (x * 32767).round().to(torch.int16)
        for j, v in enumerate(it):
            y[i + j] = v[2]
            kind.append(todo[i + j][0])
            gen.append(v[5].get("generation", "") if v[5] else "")
            dist.append(v[5].get("dist", "") if v[5] else "")
    return {"X": X, "y": y, "kind": np.array(kind), "gen": np.array(gen), "dist": np.array(dist)}

@torch.no_grad()
def predict_windows(model, X16, bs=512):
    model.eval()
    out = []
    for i in range(0, len(X16), bs):
        feat = FRONT(X16[i:i + bs].to(device).float() / 32768.0)
        with torch.autocast(device_type=device, enabled=(device == "cuda")):
            lg = model(feat.unsqueeze(1))
        out.append(F.softmax(lg.float(), 1).cpu())
    return torch.cat(out).numpy()

def val_metrics(P, V, fpr=0.005):
    """알림 대상이 아닌 창의 0.5%만 울리게 기준을 잡았을 때, 계열별로 얼마나 잡는가"""
    y, score = V["y"], P[:, ALERT].sum(1)
    pos = np.isin(y, ALERT)
    thr = float(np.quantile(score[~pos], 1 - fpr))
    rec = {CLASSES[k]: float((score[y == k] > thr).mean()) for k in ALERT if (y == k).any()}
    return {"thr": thr, "macro": float(np.mean(list(rec.values()))) if rec else 0.0, "recall": rec,
            "fa_kind": {k: float((score[V["kind"] == k] > thr).mean()) for k in ("bg", "sp", "hn", "E") if (V["kind"] == k).any()},
            "acc_pos": float((P[pos].argmax(1) == y[pos]).mean()) if pos.any() else 0.0}

# ---- 연속 소리 평가: 실제처럼 이어지는 소리에서 시간당 오경보와 재현율을 잰다 ----
def np_reverb(x, rng):
    L = int(0.6 * SR)
    h = rng.standard_normal(L) * np.exp(-6.91 * (np.arange(L) / SR) / rng.uniform(0.15, 0.9))
    h[:16] = 0
    h = h / np.sqrt((h ** 2).sum()) * 10 ** (-rng.uniform(-3, 15) / 20)
    h[0] = 1.0
    n = 1 << (len(x) + L - 1).bit_length()
    y = np.fft.irfft(np.fft.rfft(x, n) * np.fft.rfft(h, n), n)[:len(x)]
    return (y * (rms(x) / rms(y))).astype(np.float32)

def build_stream(kind, minutes, seed):
    rng = np.random.default_rng(seed)
    n, XF = int(minutes * 60 * SR), 1600
    x, pos, fin = np.zeros(n + 4 * SR, np.float32), 0, np.linspace(0, 1, 1600, dtype=np.float32)
    while pos < n:                                              # 배경: 같은 크기의 창 6~12개를 이어 붙인 장면(20%는 조용한 방)
        lvl = 10 ** (rng.uniform(-58, -30) / 20) if rng.random() < 0.8 else 0.0
        for _ in range(int(rng.integers(6, 13))):
            a = AMB_VA[int(rng.integers(len(AMB_VA)))].astype(np.float32)
            a = a / rms(a) * lvl
            a[:XF] *= fin
            a[-XF:] *= fin[::-1]
            x[pos:pos + len(a)] += a
            pos += len(a) - XF
            if pos >= n:
                break
    x = x[:n]
    byc = collections.defaultdict(list)
    for r in VA:
        byc[r["cls_id"]].append(r)
    fam = [k for k in ALERT if byc[k]]
    ev, t = [], 4.0
    while t < minutes * 60 - 12:
        u = rng.random()
        if kind == "pos":
            typ = "pos" if u < 0.6 else ("sp" if u < 0.8 else "hn")
        else:
            typ = "sp" if u < 0.45 else ("hn" if u < 0.85 or not byc[5] else "E")
        r = None
        if typ == "sp":
            w = SP_VA[int(rng.integers(len(SP_VA)))]
            L = int(rng.uniform(1.0, 3.5) * SR)
            st = int(rng.integers(0, len(w) - L + 1))
            c = w[st:st + L].astype(np.float32)
            c[:320] *= RAMP2
            c[-320:] *= RAMP2[::-1]
            s0, s1, cls = 0, L, 0
        else:
            lst = byc[fam[int(rng.integers(len(fam)))]] if typ == "pos" else (byc[5] if typ == "E" else byc[0])
            r = lst[int(rng.integers(len(lst)))]
            c, s0, s1, cls = clip16(r).astype(np.float32), r["s0"], r["s1"], r["cls_id"]
        c = c / rms(c[s0:s1])
        if rng.random() < 0.5:
            c = np_reverb(c, rng)
        lvl = rng.uniform(-42, -18)
        a = int(t * SR)
        b = min(a + len(c), n)
        x[a:b] += c[:b - a] * 10 ** (lvl / 20)
        ev.append({"type": typ, "cls": cls, "ts": t + s0 / SR, "te": t + s1 / SR, "lvl": lvl,
                   "gen": r.get("generation", "") if r else "", "dist": r.get("dist", "") if r else ""})
        t += len(c) / SR + (rng.uniform(5.0, 9.0) if kind == "pos" else rng.uniform(2.0, 6.0))
    x += rng.standard_normal(n).astype(np.float32) * 10 ** (-75 / 20)
    return np.clip(x, -1, 1), ev

@torch.no_grad()
def stream_scores(model, x):
    model.eval()
    hop = int(HOP_S * SR)
    wins = torch.from_numpy(x).to(device).unfold(0, WIN, hop)
    out = []
    for i in range(0, wins.shape[0], 512):
        feat = FRONT(wins[i:i + 512])
        with torch.autocast(device_type=device, enabled=(device == "cuda")):
            lg = model(feat.unsqueeze(1))
        out.append(F.softmax(lg.float(), 1)[:, ALERT].sum(1).cpu())
    s = torch.cat(out).numpy()
    return (np.arange(len(s)) * hop + WIN) / SR, s

def detect(tend, s, thr, smooth=3, consec=2, refr=3.0):
    """판단 규칙: 최근 3번 평균이 기준 이상인 상태가 2번 이어지면 감지, 감지 후 3초는 쉼"""
    on = np.convolve(s, np.ones(smooth) / smooth)[:len(s)] >= thr
    fires, last, run = [], -1e9, 0
    for i in range(len(on)):
        run = run + 1 if on[i] else 0
        if run >= consec and tend[i] - last >= refr:
            fires.append(tend[i])
            last = tend[i]
    return np.array(fires)

def score_stream(fires, ev):
    used, hit = np.zeros(len(fires), bool), []
    for e in ev:
        if e["type"] == "pos":
            m = (fires >= e["ts"] + 0.3) & (fires <= e["te"] + 3.25)
            hit.append(bool(m.any()))
            used |= m
    fa = collections.Counter()
    for tf in fires[~used]:
        fa[next((e["type"] for e in ev if e["type"] != "pos" and e["ts"] <= tf <= e["te"] + 3.25), "bg")] += 1
    return hit, fa

def frame_db(x):
    fr = 320
    n = len(x) // fr
    return 10 * np.log10((x[:n * fr].astype(np.float32).reshape(n, fr) ** 2).mean(1) + 1e-12)

def win_grid(n):
    hop = int(HOP_S * SR)
    return (np.arange(max(1 + (n - WIN) // hop, 0)) * hop + WIN) / SR

def gate_ok(x, tend):
    """소리 크기 게이트: 판단 시점마다 최근 1.5초 안에 바닥(하위 20%)보다 GATE_REL dB 이상이고 GATE_ABS dBFS 이상인 20ms 구간이 있어야 판단한다.
    실제 서비스의 게이트와 같은 역할(조용할 때는 모델을 돌리지 않음)."""
    e = frame_db(x)
    thr = max(float(np.percentile(e, 20)) + GATE_REL, GATE_ABS)
    c = np.concatenate([[0], np.cumsum(e >= thr)])
    i1 = np.minimum((np.asarray(tend) * SR / 320).astype(int), len(e))
    i0 = np.maximum(i1 - int(1.5 * SR / 320), 0)
    return (c[i1] - c[i0]) > 0

_GATE_CACHE = {}

THRS = [0.3, 0.5, 0.7, 0.8, 0.9, 0.93, 0.95, 0.96, 0.97, 0.975, 0.98, 0.985, 0.99, 0.995]   # 라벨 스무딩 때문에 점수가 0.96쯤에서 막혀서 윗부분을 촘촘히
THR_VER = 2                                                     # 평가 방식을 바꿀 때 올린다(저장된 모델은 학습 없이 다시 평가됨)
_UB95 = [2.996, 4.744, 6.296, 7.754, 9.154, 10.513, 11.842, 13.148, 14.434, 15.705, 16.962]

def fa_upper(n, hours):
    """오경보 n번이 관찰됐을 때, 시간당 오경보의 95% 단측 상한(포아송 가정)"""
    n = int(round(n))
    return (_UB95[n] if n < len(_UB95) else n + 1.645 * math.sqrt(n) + 1.0) / hours

def stream_eval(model):
    sc = {}
    for k in STREAMS:
        tend, s = stream_scores(model, STREAMS[k][0])
        if k not in _GATE_CACHE:
            _GATE_CACHE[k] = gate_ok(STREAMS[k][0], tend)
        sc[k] = (tend, s * _GATE_CACHE[k][:len(s)])
    hours = sum(len(STREAMS[k][0]) for k in STREAMS) / SR / 3600.0
    pos = [e for k in STREAMS for e in STREAMS[k][1] if e["type"] == "pos"]
    nE = sum(1 for k in STREAMS for e in STREAMS[k][1] if e["type"] == "E")
    table = []
    for thr in THRS:
        hits, fa = [], collections.Counter()
        for k in STREAMS:
            h, f = score_stream(detect(*sc[k], thr), STREAMS[k][1])
            hits += h
            fa.update(f)
        hits = np.array(hits)
        grp = lambda cond: float(hits[[cond(e) for e in pos]].mean()) if any(cond(e) for e in pos) else None
        table.append({"thr": thr, "fa_h": (sum(fa.values()) - fa["E"]) / hours, "recall": float(hits.mean()),
                      "by": {CLASSES[c]: grp(lambda e, c=c: e["cls"] == c) for c in ALERT},
                      "senior": grp(lambda e: e["gen"] == "senior"), "far": grp(lambda e: e["dist"] == "원거리"),
                      "quiet": grp(lambda e: e["lvl"] < -34), "fa_kind": dict(fa), "e_rate": fa["E"] / nE if nE else None})
    ok = [r for r in table if r["fa_h"] <= FA_TARGET]
    op = max(ok, key=lambda r: (r["recall"], -r["fa_h"], r["thr"])) if ok else table[-1]      # 오경보 목표 안에서 재현율이 가장 높은 기준
    return {"hours": hours, "neg_hours": sum(len(STREAMS[k][0]) for k in STREAMS) / SR / 3600.0, "n_pos": len(pos), "table": table, "op": op}

def make_scaler():
    try:
        return torch.amp.GradScaler(device, enabled=(device == "cuda"))
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

def run_dir(name):
    d = os.path.join(OUT, "runs_smoke" if SMOKE else "runs_full", name)      # 동작 확인(smoke)과 실제 학습(full)의 결과를 섞지 않는다
    os.makedirs(d, exist_ok=True)
    return d

def fmt_rec(d):
    return " ".join("%s %.2f" % (k, v) for k, v in d.items() if v is not None)

def train_run(name, arch, steps, frac=1.0, lr=2e-3, seed=SEED):
    d = run_dir(name)
    rp, cp, bp = os.path.join(d, "result.json"), os.path.join(d, "ckpt_last.pt"), os.path.join(d, "best.pt")
    sig = {"steps": steps, "frac": frac, "win": WIN, "n_train": len(TR), "batch": BATCH, "arch": arch}
    if name.startswith(VER):
        sig.update({"ver": VER, "p_real": [P_REAL_BG, P_REAL_SP], "n_real": 0 if REAL_TR is None else int(len(REAL_TR))})
    bdp = os.path.join(d, "best_dev.pt")
    use_dev = name.startswith(VER) and DEV is not None
    if os.path.exists(rp):
        old = json.load(open(rp, encoding="utf-8"))
        if all(old.get(k) == v for k, v in sig.items()):
            if old.get("thr_ver") != THR_VER and os.path.exists(bp):
                mm = ARCH[arch]().to(device)
                mm.load_state_dict(torch.load(bp, map_location=device))
                old["val"], old["stream"], old["thr_ver"] = val_metrics(predict_windows(mm, VAL["X"]), VAL), stream_eval(mm), THR_VER
                json.dump(old, open(rp, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
                print("[다시 평가] %s: 저장된 모델을 새 평가 방식(촘촘한 기준, 더 긴 소리)으로 다시 평가했습니다 (학습은 다시 하지 않음)" % name)
            print("[건너뜀] %s: 같은 설정(%d단계)으로 이미 끝난 실험" % (name, steps))
            return old
        print("[다시 학습] %s: 이전 결과는 설정이 달라서(이전 %s단계) 쓰지 않고 새로 학습합니다" % (name, old.get("steps")))
        os.replace(rp, rp + ".old")
        if os.path.exists(cp):
            os.replace(cp, cp + ".old")
    torch.manual_seed(seed)
    model, ema = ARCH[arch]().to(device), ARCH[arch]().to(device)
    ema.load_state_dict(model.state_dict())
    for p in ema.parameters():
        p.requires_grad_(False)
    opt = torch.optim.AdamW([{"params": [p for p in model.parameters() if p.ndim > 1], "weight_decay": 1e-2},
                             {"params": [p for p in model.parameters() if p.ndim <= 1], "weight_decay": 0.0}], lr=lr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
    scaler = make_scaler()
    step, best, hist, spent = 0, -1.0, [], 0.0
    best_dev, dev_best, dev_step = None, None, None
    ck = torch.load(cp, map_location=device, weights_only=False) if os.path.exists(cp) else None
    if ck is not None and not all(ck.get("sig", {}).get(k) == v for k, v in sig.items()):
        print("[주의] %s: 남아 있던 중간 저장은 설정이 달라서 쓰지 않습니다" % name)
        ck = None
    if ck is not None:
        model.load_state_dict(ck["model"]); ema.load_state_dict(ck["ema"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        step, best, hist, spent = ck["step"], ck["best"], ck["hist"], ck.get("spent", 0.0)
        best_dev = tuple(ck["best_dev"]) if ck.get("best_dev") else None
        dev_best, dev_step = ck.get("dev_best"), ck.get("dev_step")
        print("[이어서] %s: %d단계부터" % (name, step))
    ed = 1.0 - 1.0 / min(500.0, max(steps / 6.0, 10.0))
    it = iter(torch.utils.data.DataLoader(BatchGen(TR, AMB_TR, SP_TR, BATCH, seed + step, frac), batch_size=None, num_workers=WORKERS))
    t0, n0, lm = time.time(), step, None
    print("[시작] %s (%s, %d단계, 양성 데이터 %d%%)" % (name, arch, steps, int(frac * 100)))
    while step < steps:
        fg, bg, y, lf, lb = [t.to(device, non_blocking=True) for t in next(it)]
        model.train()
        with torch.no_grad():
            feat = spec_aug(FRONT(mix_aug(fg, bg, lf, lb)))
        with torch.autocast(device_type=device, enabled=(device == "cuda")):
            loss = F.cross_entropy(model(feat.unsqueeze(1)).float(), y, label_smoothing=0.05)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        with torch.no_grad():
            for pe, pm in zip(ema.parameters(), model.parameters()):
                pe.mul_(ed).add_(pm.detach(), alpha=1 - ed)
            for be, bm in zip(ema.buffers(), model.buffers()):
                be.copy_(bm)
        step += 1
        lm = loss.item() if lm is None else 0.98 * lm + 0.02 * loss.item()
        if step == n0 + 20:
            sp = (step - n0) * BATCH / (time.time() - t0)
            print("  속도 %.0f개/초 → 이 실험 남은 시간 약 %.0f분" % (sp, (steps - step) * BATCH / sp / 60))
        if step % EVAL_EVERY == 0 or step == steps:
            m = val_metrics(predict_windows(ema, VAL["X"]), VAL)
            hist.append([step, round(lm, 4), round(m["macro"], 4)])
            if m["macro"] > best:
                best = m["macro"]
                torch.save(ema.state_dict(), bp)
            print("  %5d/%d 손실 %.3f | 검증 재현율 %.3f (%s) | 오답에서 울림 %s" % (step, steps, lm, m["macro"], fmt_rec(m["recall"]), fmt_rec(m["fa_kind"])))
        if use_dev and (step % DEV_EVERY == 0 or step == steps):
            dv = dev_eval(ema)
            k = dev_key(dv)
            mark = ""
            if best_dev is None or k > best_dev:
                best_dev, dev_best, dev_step, mark = k, dv, step, "  ← 지금까지 가장 좋음(저장)"
                torch.save(ema.state_dict(), bdp)
            print("    실제 녹음 %s%s" % (dev_line(dv), mark))
        if step % CKPT_EVERY == 0 or step == steps:
            torch.save({"model": model.state_dict(), "ema": ema.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "step": step, "best": best, "hist": hist, "spent": spent + time.time() - t0, "sig": sig,
                        "best_dev": list(best_dev) if best_dev else None, "dev_best": dev_best, "dev_step": dev_step}, cp + ".tmp")
            os.replace(cp + ".tmp", cp)
    del it
    use_bdp = use_dev and os.path.exists(bdp)
    ema.load_state_dict(torch.load(bdp if use_bdp else bp, map_location=device))      # 실제 녹음으로 고른 시점(없으면 합성 검증으로 고른 시점)
    m, st = val_metrics(predict_windows(ema, VAL["X"]), VAL), stream_eval(ema)
    res = {**sig, "thr_ver": THR_VER, "name": name, "mode": MODE, "params": sum(p.numel() for p in ema.parameters()),
           "minutes": (spent + time.time() - t0) / 60.0, "val": m, "stream": st, "hist": hist,
           "ckpt": "best_dev.pt" if use_bdp else "best.pt", "dev": dev_best, "dev_step": dev_step}
    json.dump(res, open(rp, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    show(res)
    return res

def show(res):
    op, st = res["stream"]["op"], res["stream"]
    print("[결과] %s | 매개변수 %d | 학습 %.1f분" % (res["name"], res["params"], res["minutes"]))
    print("  창 검증: 재현율 %.3f (%s) | 양성 종류 맞힘 %.3f" % (res["val"]["macro"], fmt_rec(res["val"]["recall"]), res["val"]["acc_pos"]))
    print("  연속 평가(총 %.1f시간, 양성 %d개): 기준 %.3f에서 시간당 오경보 %.2f (95%% 상한 %.1f), 재현율 %.3f (%s)" % (
        st["hours"], st["n_pos"], op["thr"], op["fa_h"], fa_upper(op["fa_h"] * st["hours"], st["hours"]), op["recall"], fmt_rec(op["by"])))
    print("    노인 %s | 먼 거리 %s | 작은 소리 %s | 오경보가 난 소리 %s | E(신고 계열)에서 울린 비율 %s" % (
        op["senior"], op["far"], op["quiet"], op["fa_kind"], op["e_rate"]))
    print("    기준별: " + " / ".join("%.3f→오경보 %.1f·재현율 %.2f" % (r["thr"], r["fa_h"], r["recall"]) for r in st["table"]))

In [ ]:
# ⑤ 실제 녹음 개발용 세트(드라이브의 6_own 폴더)
#   학습용 폴더 → 학습의 배경·일상 말소리로 사용 | 응급 소리 → 재현율 | 시험용·장시간 녹음 → 시간당 오경보 | TV 위급 장면 → 따로 표시(울려도 이상하지 않은 어려운 경우)
def nfc(s):
    return unicodedata.normalize("NFC", s)

def load_audio16(p):
    r = subprocess.run(["ffmpeg", "-v", "error", "-i", p, "-ac", "1", "-ar", str(SR), "-f", "s16le", "-"], capture_output=True)
    return np.frombuffer(r.stdout, dtype="<i2").copy()

def own_root():
    for base in (ROOT, os.path.dirname(ROOT)):
        hits = glob.glob(os.path.join(base, "**", "6_own"), recursive=True)
        if hits:
            return hits[0]
    return None

# 청취로 확인한 실제 도움 요청 말(11/16): 1:47:07 작은 중얼거림, 1:49:31~1:49:50 방 밖 녹음 4번 → 오경보로 세지 않음
IGNORE = {"3시간 음성": [(6410.0, 6445.0), (6560.0, 6600.0)]}

def kind_of(rel):
    parts = rel.split(os.sep)
    d, f = parts[0], parts[-1]
    if "학습용" in d:
        return "train"
    if "응급" in d:
        return "pos"
    if "위급 장면" in f:
        return "tv"
    if "시험용" in d or "장시간" in d:
        return "neg"
    return None

def family_of(f):
    if "도와" in f:
        return "A"
    if "사람 살려" in f:
        return "C"
    if "살려" in f or "구해" in f:
        return "B"
    if "119" in f:
        return "D"
    return "?"

def utter_segments(x):
    """말 구간: 20ms 크기가 바닥(하위 20%)보다 12dB 이상이면 말, 0.6초 이내 틈은 한 덩어리, 0.3초 미만은 버림"""
    e = frame_db(x)
    n = len(e)
    thr = max(float(np.percentile(e, 20)) + 12.0, -70.0)
    on = e > thr
    segs, i = [], 0
    while i < n:
        if on[i]:
            j = i
            while j < n and (on[j] or on[j + 1:j + 31].any()):
                j += 1
            if j - i >= 15:
                segs.append((i * 320 / SR, j * 320 / SR))
            i = j
        else:
            i += 1
    return segs

def load_dev():
    root = own_root()
    if root is None:
        print("6_own 폴더를 찾지 못했습니다 → 실제 녹음 없이 합성 점수로만 고릅니다")
        return None, None
    files = sorted(p for p in glob.glob(os.path.join(root, "**", "*"), recursive=True) if p.lower().endswith((".m4a", ".wav", ".mp3")))
    dev, tw = {"pos": [], "neg": [], "tv": []}, []
    for p in files:
        rel = nfc(os.path.relpath(p, root))
        k = kind_of(rel)
        if k is None:
            continue
        x = load_audio16(p)
        if len(x) < WIN:
            continue
        if k == "train":
            n = len(x) // (4 * SR)
            if n:
                tw.append(x[:n * 4 * SR].reshape(n, 4 * SR))
            continue
        xf = x.astype(np.float32) / 32768.0
        it = {"name": rel, "x": x, "gate": gate_ok(xf, win_grid(len(x)))}
        if k == "pos":
            it["segs"], it["fam"] = utter_segments(xf), family_of(os.path.basename(rel))
        if k == "neg":
            it["ignore"] = next((v for key, v in IGNORE.items() if key in rel), [])
        dev[k].append(it)
    real = np.concatenate(tw) if tw else None
    fam = collections.Counter()
    for it in dev["pos"]:
        fam[it["fam"]] += len(it["segs"])
    hrs = sum(len(it["x"]) / SR - sum(b - a for a, b in it["ignore"]) for it in dev["neg"]) / 3600.0
    print("실제 녹음: 응급 소리 %d개 파일(말 구간 %s) | 오경보 측정 %.2f시간(%d개 파일) | TV 위급 장면 %d개 | 학습에 쓸 실제 녹음 %.1f분" % (
        len(dev["pos"]), dict(fam), hrs, len(dev["neg"]), len(dev["tv"]), 0 if real is None else len(real) * 4 / 60.0))
    for it in dev["pos"]:
        print("  응급: %-36s 말 구간 %2d개 (계열 %s)" % (it["name"][:36], len(it["segs"]), it["fam"]))
    return dev, real

def dev_eval(model):
    """실제 녹음에서 기준별 재현율(말 구간)과 시간당 오경보. 게이트를 적용한다"""
    hits = {t: collections.Counter() for t in THRS}
    pf = {t: {} for t in THRS}
    tot = collections.Counter()
    for it in DEV["pos"]:
        tend, s = stream_scores(model, it["x"].astype(np.float32) / 32768.0)
        s = s * it["gate"][:len(s)]
        tot[it["fam"]] += len(it["segs"])
        for t in THRS:
            f = detect(tend, s, t)
            h = sum(int(((f >= a + 0.3) & (f <= b + 3.25)).any()) for a, b in it["segs"])
            hits[t][it["fam"]] += h
            pf[t][it["name"]] = "%d/%d" % (h, len(it["segs"]))
    fa, tv, hours = collections.Counter(), collections.Counter(), 0.0
    for key in ("neg", "tv"):
        for it in DEV[key]:
            tend, s = stream_scores(model, it["x"].astype(np.float32) / 32768.0)
            s = s * it["gate"][:len(s)]
            ign = it.get("ignore", [])
            if key == "neg":
                hours += len(it["x"]) / SR / 3600.0 - sum(b - a for a, b in ign) / 3600.0
            for t in THRS:
                f = [v for v in detect(tend, s, t) if not any(a <= v <= b for a, b in ign)]
                (fa if key == "neg" else tv)[t] += len(f)
    npos = sum(tot.values())
    table = [{"thr": t, "recall": sum(hits[t].values()) / max(npos, 1), "hit": sum(hits[t].values()),
              "by": {k: hits[t][k] / tot[k] for k in sorted(tot)}, "fa_n": fa[t], "fa_h": fa[t] / max(hours, 1e-9),
              "tv": tv[t], "per_file": pf[t]} for t in THRS]
    ok = [r for r in table if r["fa_h"] <= DEV_FA_TARGET]
    op = max(ok, key=lambda r: (r["recall"], -r["fa_h"], r["thr"])) if ok else min(table, key=lambda r: (r["fa_h"], -r["recall"]))
    return {"table": table, "op": op, "n_pos": npos, "hours": hours}

def dev_key(d):
    op = d["op"]
    return (1 if op["fa_h"] <= DEV_FA_TARGET else 0, round(op["recall"], 4), -round(op["fa_h"], 4), op["thr"])   # 같으면 기준이 높은(여유가 큰) 쪽

def dev_line(d):
    op = d["op"]
    return "기준 %.3f: 재현 %d/%d (%.0f%%; %s) | 오경보 %d번/%.1f시간 = %.2f/시간 (95%% 상한 %.1f) | TV 위급 장면 울림 %d번" % (
        op["thr"], op["hit"], d["n_pos"], 100 * op["recall"], fmt_rec(op["by"]), op["fa_n"], d["hours"], op["fa_h"],
        fa_upper(op["fa_n"], d["hours"]), op["tv"])

DEV_SIG = None
def eval_saved(name, arch, ck="best.pt"):
    """이미 학습된 모델을 실제 녹음으로 평가(학습 없음). 같은 평가 조건이면 저장된 결과를 다시 씀"""
    p = os.path.join(run_dir(name), "dev_%s.json" % VER)
    if os.path.exists(p):
        old = json.load(open(p, encoding="utf-8"))
        if old.get("sig") == DEV_SIG:
            return old["dev"]
    m = ARCH[arch]().to(device)
    m.load_state_dict(torch.load(os.path.join(run_dir(name), ck), map_location=device))
    d = dev_eval(m)
    json.dump({"sig": DEV_SIG, "dev": d}, open(p, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    return d

DEV, REAL_TR = load_dev()
if DEV is not None:
    DEV_SIG = {"n_pos": sum(len(it["segs"]) for it in DEV["pos"]), "neg_s": int(sum(len(it["x"]) for it in DEV["neg"]) / SR),
               "gate": [GATE_REL, GATE_ABS], "thrs": THRS, "ignore": IGNORE}

In [ ]:
# ⑤-2 Pi 마이크 녹음(드라이브 LAB/own_pi/own_test_*.tar.gz, rec_session.py pack 결과)
#   R(학습용) → A~D 양성에 섞음 | C(보정) → ⑥-2에서 점검만 | P·N(시험용) → 읽지 않음(최종 평가 전용)
import tarfile, wave
PI_DIR = os.path.join(LOCAL, "own_pi")

def load_pi():
    tars = sorted(glob.glob(os.path.join(ROOT, "own_pi", "own_test_*.tar.gz")))
    tars = [t for t in tars if not t.endswith("_long.tar.gz")]
    if not tars:
        print("LAB/own_pi에 묶음 파일이 없습니다 → R 없이 진행(2차와 같은 조건)")
        return None, None
    tp = tars[-1]                                                  # 가장 최근 묶음(이전 녹음도 모두 들어 있음)
    if os.path.isdir(PI_DIR):
        shutil.rmtree(PI_DIR)
    os.makedirs(PI_DIR)
    with tarfile.open(tp) as tf:
        tf.extractall(PI_DIR)
    base = os.path.join(PI_DIR, "own_test")
    labs = read_csv(os.path.join(base, "labels.csv"))
    print("Pi 녹음 묶음: %s | 발화 %d개" % (os.path.basename(tp), len(labs)))
    return base, labs

def wav_ch0(p):
    with wave.open(p) as w:
        ch, raw = w.getnchannels(), w.readframes(w.getnframes())
    return np.frombuffer(raw, dtype="<i2").reshape(-1, ch)[:, 0].copy()

PI_BASE, PI_LABS = load_pi()
RPOS, PI_FILES = {k: [] for k in "ABCD"}, {}
PI_LEVELS = collections.defaultdict(list)                          # 코드별 말소리 최대 크기(dBFS, 20ms) — 서비스 게이트 판단용
if PI_LABS:
    for r in PI_LABS:
        f = r["file"]
        if f not in PI_FILES:
            PI_FILES[f] = wav_ch0(os.path.join(PI_BASE, f))
        x = PI_FILES[f]
        a, b = int(float(r["start_s"]) * SR), int(float(r["end_s"]) * SR)
        if r["timing"] == "detected":
            PI_LEVELS[r["code"]].append(float(frame_db(x[a:b].astype(np.float32) / 32768.0).max()))
        if not r["code"].startswith("R") or r["timing"] != "detected" or r["family"] not in RPOS:
            continue
        o = max(0, a - int(2.5 * SR))                               # 앞 2.5초·뒤 1.3초 실제 방 소리를 같이 씀(앞뒤 말과는 4초 이상 떨어져 있음)
        n = min(len(x), b + int(1.3 * SR)) - o
        RPOS[r["family"]].append({"arr": x, "o": o, "n": n, "s0": a - o, "s1": b - o, "cls_id": "ABCD".index(r["family"]) + 1,
                                  "role": "pi_R", "note": r["text"], "stem": r["code"], "md5": "pi%s_%s" % (f, r["idx"])})
RPOS = {k: [r for r in v if fits(r)] for k, v in RPOS.items()}
RPOS_INFO = ", ".join("%s %d" % (k, len(v)) for k, v in RPOS.items()) if any(RPOS.values()) else "없음"
print("R 양성(계열별): %s | 양성 중 R에서 뽑는 비율 %.2f" % (RPOS_INFO, P_REAL_POS))
for c in sorted(PI_LEVELS):
    v = np.array(PI_LEVELS[c])
    print("  %-14s 말 %3d개 | 최대 크기 중앙 %.1f dBFS (하위 10%% %.1f) | -43dBFS 넘는 비율 %.0f%%" % (c, len(v), np.median(v), np.percentile(v, 10), 100 * (v >= -43).mean()))

_gen_item_base = gen_item
def gen_item(kind, P, AMB, SPW, rng, train=True, row=None):
    """A~D 양성의 일부를 Pi 실제 녹음(R)으로 바꿔 뽑는다. 나머지 처리(빠르기·위치·배경·크기)는 2차와 같다"""
    if train and row is None and kind in RPOS and RPOS[kind] and rng.random() < P_REAL_POS:
        row = RPOS[kind][int(rng.integers(len(RPOS[kind])))]
    return _gen_item_base(kind, P, AMB, SPW, rng, train, row)


In [ ]:
# ⑥ 실행: 1차 모델을 실제 녹음으로 다시 평가 → 2차 학습(실제 녹음 섞기, 실제 녹음으로 가장 좋은 시점 저장) → 가장 좋은 모델 고르기
t_all = time.time()
VAL = build_val()
STREAMS = {"neg": build_stream("neg", NEG_MIN, 11), "pos": build_stream("pos", POS_MIN, 12)}
print("합성 검증 창 %d개 | 합성 연속 평가 %.0f분" % (len(VAL["y"]), NEG_MIN + POS_MIN))

ALLM = {}
if DEV is not None:
    for nm, ar in [("r2_tcresnet", "tcresnet"), ("s2_tcresnet", "tcresnet"), ("s1_tcresnet", "tcresnet"), ("s1_bcresnet", "bcresnet"), ("s1_crnn", "crnn"), ("s1_dscnn", "dscnn")]:
        ck = "best_dev.pt" if os.path.exists(os.path.join(run_dir(nm), "best_dev.pt")) else "best.pt"
        if os.path.exists(os.path.join(run_dir(nm), ck)):
            d = eval_saved(nm, ar, ck)
            ALLM[nm] = (ar, ck, d)
            print("[이전 모델 재평가] %-14s %s" % (nm, dev_line(d)))
for a in CANDS2:
    nm = "%s_%s" % (VER, a)
    r = train_run(nm, a, R2_STEPS)
    if r.get("dev"):
        ALLM[nm] = (a, r["ckpt"], r["dev"])
        print("[3차 결과] %-14s %s (가장 좋은 시점 %s단계)" % (nm, dev_line(r["dev"]), r.get("dev_step")))

print("\n================ 요약 (이 아래를 복사해서 보내 주세요) ================")
print("Pi 학습 녹음(R): %s | 양성 중 R 비율 %.2f" % (RPOS_INFO, P_REAL_POS))
print("데이터: 학습 %d클립 + 실제 녹음 %.1f분(학습용) | MODE %s | device %s | 게이트 %.0fdB/%.0fdBFS | 실제 녹음 오경보 목표 %.1f/시간" % (
    len(TR), 0 if REAL_TR is None else len(REAL_TR) * 4 / 60.0, MODE, device, GATE_REL, GATE_ABS, DEV_FA_TARGET))
if ALLM:
    FINAL = max(ALLM, key=lambda k: dev_key(ALLM[k][2]))
    print("실제 녹음 개발용 세트: 응급 소리 말 %d개, 오경보 측정 %.2f시간 (참고: 같은 녹음에서 Whisper 65/73, 오경보 0)" % (ALLM[FINAL][2]["n_pos"], ALLM[FINAL][2]["hours"]))
    for k in sorted(ALLM, key=lambda k: dev_key(ALLM[k][2]), reverse=True):
        print("  %-14s %s%s" % (k, dev_line(ALLM[k][2]), "  ← 선택" if k == FINAL else ""))
    d = ALLM[FINAL][2]
    print("선택한 모델 %s의 기준별 실제 결과:" % FINAL)
    for r in d["table"]:
        if r["thr"] >= 0.5:
            print("  %.3f → 재현 %d/%d (%s) | 오경보 %d번 (%.2f/시간) | TV 위급 장면 %d번" % (r["thr"], r["hit"], d["n_pos"], fmt_rec(r["by"]), r["fa_n"], r["fa_h"], r["tv"]))
    print("  기준 %.3f에서 파일별: %s" % (d["op"]["thr"], " | ".join("%s %s" % (os.path.basename(k)[:16], v) for k, v in d["op"]["per_file"].items())))
    BEST, BEST_CK = ALLM[FINAL][0], ALLM[FINAL][1]
else:
    FINAL = "%s_%s" % (VER, CANDS2[0])
    BEST, BEST_CK = CANDS2[0], "best.pt"
    print("실제 녹음이 없어서 합성 점수 기준으로 %s를 씁니다." % FINAL)
print("전체 걸린 시간 %.1f분" % ((time.time() - t_all) / 60))

In [ ]:
# ⑥-2 Pi 보정 녹음(C) 점검: 선택한 모델로 C1·C2에서 몇 개 잡는지, C0(조용한 방)에서 울리는지. 기준·게이트 조정 참고용
if PI_LABS:
    m = ARCH[BEST]().to(device)
    m.load_state_dict(torch.load(os.path.join(run_dir(FINAL), BEST_CK), map_location=device))
    thr = ALLM[FINAL][2]["op"]["thr"] if FINAL in ALLM else 0.9
    by = collections.defaultdict(list)
    for r in PI_LABS:
        if r["code"].startswith("C"):
            by[r["file"]].append(r)
    cfiles = sorted(os.path.relpath(q, PI_BASE) for q in glob.glob(os.path.join(PI_BASE, "audio", "C*", "*.wav")))
    for f in cfiles:
        x = PI_FILES.get(f)
        if x is None:
            x = PI_FILES[f] = wav_ch0(os.path.join(PI_BASE, f))
        xf = x.astype(np.float32) / 32768.0
        tend, s = stream_scores(m, xf)
        s = s * gate_ok(xf, tend)[:len(s)]
        for t in sorted({thr, 0.5, 0.9}):
            fires = detect(tend, s, t)
            segs = [(float(r["start_s"]), float(r["end_s"])) for r in by.get(f, [])]
            hit = sum(int(((fires >= a + 0.3) & (fires <= b + 3.25)).any()) for a, b in segs)
            extra = sum(1 for v in fires if not any(a + 0.3 <= v <= b + 3.25 for a, b in segs))
            print("  %-34s 기준 %.3f → 잡음 %d/%d | 말 아닌 곳에서 울림 %d번%s" % (os.path.basename(f), t, hit, len(segs), extra, "  ← 선택 기준" if t == thr else ""))
        if by.get(f):
            print("    놓친 말: %s" % (", ".join(r["text"] for r in by[f] if not ((detect(tend, s, thr) >= float(r["start_s"]) + 0.3) & (detect(tend, s, thr) <= float(r["end_s"]) + 3.25)).any()) or "없음"))
else:
    print("Pi 녹음이 없어 건너뜀")


In [ ]:
# ⑦ ONNX로 내보내기 + 검산 (파이에서 쓸 파일: 모델, 설정, mel 필터)
if "FINAL" not in globals():
    raise RuntimeError("⑥번 셀까지 먼저 실행해야 합니다. 런타임이 초기화됐다면 '모두 실행'을 다시 하세요(끝난 학습은 건너뜁니다).")

def export(name, arch, ck):
    try:
        import onnxruntime as ort, onnx
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=True)
        import onnxruntime as ort
    d = run_dir(name)
    model = ARCH[arch]()
    model.load_state_dict(torch.load(os.path.join(d, ck), map_location="cpu"))
    model.eval()
    path = os.path.join(d, name + ".onnx")
    kw = dict(input_names=["feat"], output_names=["logits"], dynamic_axes={"feat": {0: "batch"}}, opset_version=17)
    dummy = torch.zeros(1, 1, N_MELS, N_FRAMES)
    try:
        torch.onnx.export(model, dummy, path, dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(model, dummy, path, **kw)
    thr = ALLM[name][2]["op"]["thr"] if name in ALLM else 0.9
    np.save(os.path.join(d, "mel_fb.npy"), FB)
    json.dump({"sr": SR, "win": WIN, "n_fft": N_FFT, "hop": HOP, "n_mels": N_MELS, "fmin": FMIN, "fmax": FMAX, "classes": CLASSES,
               "alert": ALERT, "hop_s": HOP_S, "thr": thr, "smooth": 3, "consec": 2, "refractory_s": 3.0, "arch": arch, "ckpt": ck,
               "gate_rel_db": GATE_REL, "gate_abs_dbfs": GATE_ABS, "gate_window_s": 1.5, "feature": "log-mel, 채널별 시간 평균 제거, 전체 RMS로 나눔"},
              open(os.path.join(d, name + "_config.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    x = VAL["X"][:16].float().numpy() / 32768.0
    f_np = np.stack([np_features(v) for v in x])
    f_t = Frontend()(torch.from_numpy(x)).numpy()
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    lg_o = sess.run(None, {"feat": f_np[:, None]})[0]
    with torch.no_grad():
        lg_t = model(torch.from_numpy(f_t)[:, None]).numpy()
    t0 = time.time()
    for _ in range(20):
        sess.run(None, {"feat": f_np[:1, None]})
    print("[내보내기] %s (%s) | 기준 %.3f | 특징 차이 최대 %.5f | 점수 차이 최대 %.5f | 이 컴퓨터에서 한 번 판단 %.1fms | 파일 %.0fKB" % (
        name, ck, thr, np.abs(f_np - f_t).max(), np.abs(lg_o - lg_t).max(), (time.time() - t0) / 20 * 1000, os.path.getsize(path) / 1024))
    print("  파이로 옮길 파일: %s" % ", ".join(os.path.join(d, n) for n in (name + ".onnx", name + "_config.json", "mel_fb.npy")))
    return path

export(FINAL, BEST, BEST_CK)